# Huấn luyện BPR trên Google Colab

Notebook này chạy toàn bộ quy trình cho **BPR – Bayesian Personalized Ranking (xếp hạng cá nhân hóa theo cặp)** của project `recsys-hust`:

1. Gắn Google Drive.
2. Khai báo đường dẫn và tham số.
3. Giải nén project từ Drive sang `/content`.
4. Cài thư viện của project.
5. Kiểm tra GPU và dữ liệu MINDlarge.
6. Kiểm tra thử cách tạo một lô BPR pair.
7. Train mới hoặc chạy tiếp từ checkpoint.
8. Tự sao lưu checkpoint về Google Drive sau mỗi epoch.
9. Xem lịch sử huấn luyện.
10. Đánh giá lại `best.pt` trên toàn bộ tập `dev`.
11. Kiểm tra và lưu kết quả cuối cùng.

> **Trước khi chạy:** trong Colab chọn `Runtime -> Change runtime type -> GPU`.

Notebook giả sử file ZIP trên Google Drive đã là **project mới nhất**, tức phần BPR có cấu trúc:

```text
src/models/bpr/
├── __init__.py
├── data.py
├── model.py
├── train.py
└── evaluate.py
```


## 1. Gắn Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 2. Cấu hình

Bạn chủ yếu cần sửa `PROJECT_ZIP_DRIVE` nếu file ZIP nằm ở vị trí khác.

### Chạy mới

Giữ:

```python
MODE = "new"
```

### Chạy tiếp từ checkpoint

Đổi thành:

```python
MODE = "resume"
RESUME_FILENAME = "epoch_003.pt"
```


In [ ]:
from pathlib import Path

# ============================================================
# ĐƯỜNG DẪN
# ============================================================

# File ZIP project trên Google Drive.
PROJECT_ZIP_DRIVE = Path(
    "/content/drive/MyDrive/recsys/recsys-hust-main.zip"
)

# Project sẽ được giải nén và chạy tại ổ cục bộ của Colab.
PROJECT_DIR = Path("/content/recsys-hust-main")

# Nơi lưu checkpoint BPR lâu dài trên Google Drive.
DRIVE_BPR_CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/recsys/checkpoints/bpr"
)

# ============================================================
# CHẾ ĐỘ HUẤN LUYỆN
# ============================================================

# "new"    : train lại từ đầu.
# "resume" : chạy tiếp từ một checkpoint.
MODE = "new"

# Số epoch chạy trong LẦN CHẠY NÀY.
# Ví dụ resume từ epoch 3 và để 3 thì sẽ chạy epoch 4, 5, 6.
EPOCHS_TO_RUN = 3

# Chỉ dùng khi MODE = "resume".
RESUME_FILENAME = "epoch_003.pt"

# Khi train mới, xóa checkpoint BPR cũ trên Drive để tránh lẫn kết quả.
# Đổi thành False nếu bạn muốn tự quản lý thư mục cũ.
CLEAR_DRIVE_CHECKPOINTS_WHEN_NEW = True

# ============================================================
# THAM SỐ BPR
# ============================================================

EMBEDDING_DIM = 64
BATCH_SIZE = 4096
SHUFFLE_BUFFER_SIZE = 65536
READ_BATCH_SIZE = 16384
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
SEED = 42

# Đánh giá dev theo lô 1000 impression.
EVAL_BATCH_SIZE = 1000

print("PROJECT_ZIP_DRIVE:", PROJECT_ZIP_DRIVE)
print("PROJECT_DIR:", PROJECT_DIR)
print("DRIVE_BPR_CHECKPOINT_DIR:", DRIVE_BPR_CHECKPOINT_DIR)
print("MODE:", MODE)
print("EPOCHS_TO_RUN:", EPOCHS_TO_RUN)


## 3. Kiểm tra file ZIP và giải nén project vào `/content`

In [ ]:
import shutil
import zipfile

if not PROJECT_ZIP_DRIVE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy file project trên Drive: {PROJECT_ZIP_DRIVE}"
    )

# Xóa bản project cũ ở /content để lần chạy này sạch sẽ.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

extract_root = Path("/content/_recsys_extract")

if extract_root.exists():
    shutil.rmtree(extract_root)

extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP_DRIVE, "r") as zip_file:
    zip_file.extractall(extract_root)

# Tìm thư mục gốc thật sự của project bằng pyproject.toml.
pyproject_files = list(extract_root.rglob("pyproject.toml"))

if len(pyproject_files) == 0:
    raise FileNotFoundError(
        "Không tìm thấy pyproject.toml trong ZIP. "
        "Hãy kiểm tra lại file project."
    )

if len(pyproject_files) > 1:
    print("Tìm thấy nhiều pyproject.toml:")
    for path in pyproject_files:
        print(" -", path)

project_source = pyproject_files[0].parent

shutil.move(
    str(project_source),
    str(PROJECT_DIR),
)

# Dọn thư mục giải nén tạm.
if extract_root.exists():
    shutil.rmtree(extract_root)

print("Đã giải nén project tại:")
print(PROJECT_DIR)


## 4. Kiểm tra cấu trúc BPR

In [ ]:
required_files = [
    PROJECT_DIR / "pyproject.toml",
    PROJECT_DIR / "src/models/bpr/__init__.py",
    PROJECT_DIR / "src/models/bpr/data.py",
    PROJECT_DIR / "src/models/bpr/model.py",
    PROJECT_DIR / "src/models/bpr/train.py",
    PROJECT_DIR / "src/models/bpr/evaluate.py",
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    print("Thiếu các file:")
    for path in missing_files:
        print(" -", path)

    raise FileNotFoundError(
        "Project chưa có đầy đủ cấu trúc BPR."
    )

print("Cấu trúc BPR hợp lệ.")

for path in required_files:
    print("✓", path.relative_to(PROJECT_DIR))


## 5. Cài project và thư viện

In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT_DIR)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
    ],
    check=True,
)

print("Đã cài project.")
print("Thư mục làm việc:", Path.cwd())


## 6. Kiểm tra GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "Colab chưa cấp GPU. "
        "Hãy chọn Runtime -> Change runtime type -> GPU rồi chạy lại."
    )


## 7. Kiểm tra dữ liệu MINDlarge

In [ ]:
import pyarrow.dataset as ds

train_dir = (
    PROJECT_DIR
    / "data/processed/mind_large/train/pointwise"
)

dev_dir = (
    PROJECT_DIR
    / "data/processed/mind_large/dev/impressions"
)

user_mapping_dir = (
    PROJECT_DIR
    / "data/processed/mind_large/mappings/users"
)

item_mapping_dir = (
    PROJECT_DIR
    / "data/processed/mind_large/mappings/items"
)

required_data_dirs = [
    train_dir,
    dev_dir,
    user_mapping_dir,
    item_mapping_dir,
]

for path in required_data_dirs:
    if not path.exists():
        raise FileNotFoundError(
            f"Thiếu dữ liệu: {path}"
        )

num_train_rows = ds.dataset(
    train_dir,
    format="parquet",
).count_rows()

num_dev_impressions = ds.dataset(
    dev_dir,
    format="parquet",
).count_rows()

num_users = ds.dataset(
    user_mapping_dir,
    format="parquet",
).count_rows()

num_items = ds.dataset(
    item_mapping_dir,
    format="parquet",
).count_rows()

print("Số dòng pointwise train:", num_train_rows)
print("Số impression dev:", num_dev_impressions)
print("Số user:", num_users)
print("Số item:", num_items)


## 8. Kiểm tra thử một lô BPR pair

BPR đọc trực tiếp `train/pointwise`, gom lại theo `impression_id`, rồi biến dữ liệu thành:

```text
(user_idx, positive_item_idx, negative_item_idx)
```

Không lấy thêm negative mới từ toàn bộ catalog.


In [ ]:
from src.models.bpr.data import iter_bpr_batches

sample_batch = next(
    iter_bpr_batches(
        train_dir,
        batch_size=8,
        shuffle=False,
        shuffle_buffer_size=8,
        read_batch_size=READ_BATCH_SIZE,
    )
)

sample_users, sample_positive, sample_negative = sample_batch

print("user_idx       :", sample_users.tolist())
print("positive items :", sample_positive.tolist())
print("negative items :", sample_negative.tolist())
print("Số BPR pair    :", len(sample_users))


## 9. Chuẩn bị checkpoint

Checkpoint được train trong `/content` để tốc độ đọc/ghi tốt hơn.

Sau mỗi epoch hoàn thành, notebook sẽ tự sao lưu:

```text
artifacts/checkpoints/bpr/
```

sang Google Drive.

Nếu chạy `resume`, notebook sẽ chép toàn bộ checkpoint cũ từ Drive về project trước để giữ cả `history.csv` và `best.pt`.


In [ ]:
LOCAL_BPR_CHECKPOINT_DIR = (
    PROJECT_DIR
    / "artifacts/checkpoints/bpr"
)

if MODE not in {"new", "resume"}:
    raise ValueError(
        'MODE chỉ được là "new" hoặc "resume".'
    )

if MODE == "new":
    if (
        CLEAR_DRIVE_CHECKPOINTS_WHEN_NEW
        and DRIVE_BPR_CHECKPOINT_DIR.exists()
    ):
        shutil.rmtree(
            DRIVE_BPR_CHECKPOINT_DIR
        )

    DRIVE_BPR_CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("Chế độ TRAIN MỚI.")
    print(
        "Checkpoint mới sẽ được sao lưu vào:",
        DRIVE_BPR_CHECKPOINT_DIR,
    )

else:
    if not DRIVE_BPR_CHECKPOINT_DIR.exists():
        raise FileNotFoundError(
            "Không tìm thấy thư mục checkpoint trên Drive: "
            f"{DRIVE_BPR_CHECKPOINT_DIR}"
        )

    if LOCAL_BPR_CHECKPOINT_DIR.exists():
        shutil.rmtree(
            LOCAL_BPR_CHECKPOINT_DIR
        )

    shutil.copytree(
        DRIVE_BPR_CHECKPOINT_DIR,
        LOCAL_BPR_CHECKPOINT_DIR,
    )

    resume_local = (
        LOCAL_BPR_CHECKPOINT_DIR
        / RESUME_FILENAME
    )

    if not resume_local.exists():
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint resume: {resume_local}"
        )

    print("Chế độ RESUME.")
    print("Checkpoint:", resume_local)


## 10. Huấn luyện BPR

Lệnh dưới đây truyền rõ các tham số để lần chạy có thể tái lập.

Không truyền `--eval-max-impressions`, vì vậy sau mỗi epoch mô hình sẽ được đánh giá trên **toàn bộ tập dev**.

Notebook theo dõi đầu ra của quá trình train. Khi một epoch hoàn thành, checkpoint và `history.csv` sẽ tự được sao lưu sang Google Drive.


In [ ]:
import os
import subprocess
import sys
import shutil


def sync_checkpoints_to_drive():
    """Sao lưu checkpoint BPR hiện có sang Google Drive."""
    if not LOCAL_BPR_CHECKPOINT_DIR.exists():
        return

    DRIVE_BPR_CHECKPOINT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        LOCAL_BPR_CHECKPOINT_DIR,
        DRIVE_BPR_CHECKPOINT_DIR,
        dirs_exist_ok=True,
    )


command = [
    sys.executable,
    "-m",
    "src.models.bpr.train",
    "--epochs",
    str(EPOCHS_TO_RUN),
    "--embedding-dim",
    str(EMBEDDING_DIM),
    "--batch-size",
    str(BATCH_SIZE),
    "--shuffle-buffer-size",
    str(SHUFFLE_BUFFER_SIZE),
    "--read-batch-size",
    str(READ_BATCH_SIZE),
    "--learning-rate",
    str(LEARNING_RATE),
    "--weight-decay",
    str(WEIGHT_DECAY),
    "--seed",
    str(SEED),
    "--eval-batch-size",
    str(EVAL_BATCH_SIZE),
]

if MODE == "resume":
    command.extend(
        [
            "--resume",
            str(
                LOCAL_BPR_CHECKPOINT_DIR
                / RESUME_FILENAME
            ),
        ]
    )

print("Lệnh train:")
print(" ".join(command))
print()

environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    command,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=environment,
)

for line in process.stdout:
    print(line, end="")

    # Các dòng này xuất hiện sau khi history/best của epoch
    # đã được xử lý xong, nên đây là thời điểm tốt để sao lưu.
    if (
        "Đã cập nhật mô hình tốt nhất." in line
        or "Epoch hiện tại không tốt hơn" in line
    ):
        sync_checkpoints_to_drive()
        print(
            ">>> Đã sao lưu checkpoint hiện tại lên Google Drive."
        )

return_code = process.wait()

# Sao lưu lần cuối dù train thành công hay dừng ở cuối quá trình.
sync_checkpoints_to_drive()

if return_code != 0:
    raise RuntimeError(
        f"Quá trình train kết thúc với mã lỗi {return_code}."
    )

print()
print("Train hoàn tất.")
print(
    "Checkpoint đã được lưu tại:",
    DRIVE_BPR_CHECKPOINT_DIR,
)


## 11. Xem lịch sử huấn luyện

In [ ]:
import pandas as pd
from IPython.display import display

history_path = (
    LOCAL_BPR_CHECKPOINT_DIR
    / "history.csv"
)

if not history_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy history.csv: {history_path}"
    )

history_df = pd.read_csv(
    history_path
)

display(history_df)

best_row = history_df.loc[
    history_df["ndcg_10"].idxmax()
]

print()
print("===== EPOCH TỐT NHẤT TRONG HISTORY =====")
print("Epoch:", int(best_row["epoch"]))
print(f"Loss train: {best_row['train_loss']:.6f}")
print(f"MRR: {best_row['mrr']:.6f}")
print(f"HitRate@5: {best_row['hit_rate_5']:.6f}")
print(f"HitRate@10: {best_row['hit_rate_10']:.6f}")
print(f"Recall@5: {best_row['recall_5']:.6f}")
print(f"Recall@10: {best_row['recall_10']:.6f}")
print(f"NDCG@5: {best_row['ndcg_5']:.6f}")
print(f"NDCG@10: {best_row['ndcg_10']:.6f}")


## 12. Kiểm tra `best.pt`

In [ ]:
best_checkpoint_path = (
    LOCAL_BPR_CHECKPOINT_DIR
    / "best.pt"
)

if not best_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy best.pt: {best_checkpoint_path}"
    )

checkpoint = torch.load(
    best_checkpoint_path,
    map_location="cpu",
)

print("Epoch:", checkpoint.get("epoch"))
print("Embedding dim:", checkpoint.get("embedding_dim"))
print("Use item bias:", checkpoint.get("use_item_bias"))
print("Train loss:", checkpoint.get("train_loss"))
print("MRR:", checkpoint.get("mrr"))
print("HitRate@5:", checkpoint.get("hit_rate_5"))
print("HitRate@10:", checkpoint.get("hit_rate_10"))
print("Recall@5:", checkpoint.get("recall_5"))
print("Recall@10:", checkpoint.get("recall_10"))
print("NDCG@5:", checkpoint.get("ndcg_5"))
print("NDCG@10:", checkpoint.get("ndcg_10"))
print("Best epoch:", checkpoint.get("best_epoch"))
print("Best NDCG@10:", checkpoint.get("best_ndcg_10"))


## 13. Đánh giá lại mô hình tốt nhất trên toàn bộ `dev`

Bước này không train lại. Nó chỉ tải `best.pt` rồi tính lại:

- `MRR`
- `HitRate@5`
- `HitRate@10`
- `Recall@5`
- `Recall@10`
- `NDCG@5`
- `NDCG@10`

trên toàn bộ các trường hợp **warm-start – user/item đã xuất hiện trong train**.


In [ ]:
evaluate_command = [
    sys.executable,
    "-m",
    "src.models.bpr.evaluate",
    "--checkpoint",
    str(best_checkpoint_path),
    "--batch-size",
    str(EVAL_BATCH_SIZE),
]

subprocess.run(
    evaluate_command,
    cwd=PROJECT_DIR,
    check=True,
)


## 14. Sao lưu lần cuối lên Google Drive

In [ ]:
sync_checkpoints_to_drive()

print("Đã sao lưu toàn bộ:")
for path in sorted(
    DRIVE_BPR_CHECKPOINT_DIR.glob("*")
):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(
            f"- {path.name}: {size_mb:.2f} MB"
        )


## 15. Tạo file ZIP checkpoint trên Google Drive

Bước này không bắt buộc, nhưng tiện nếu muốn tải toàn bộ checkpoint về máy một lần.


In [ ]:
archive_base = (
    DRIVE_BPR_CHECKPOINT_DIR.parent
    / "bpr_checkpoints"
)

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=DRIVE_BPR_CHECKPOINT_DIR,
)

print("Đã tạo:")
print(archive_path)


# Chạy tiếp trong một phiên Colab khác

Lần sau chỉ cần:

1. Đưa project ZIP mới nhất lên Drive.
2. Giữ thư mục checkpoint BPR trên Drive.
3. Trong ô cấu hình đổi:

```python
MODE = "resume"
EPOCHS_TO_RUN = 3
RESUME_FILENAME = "epoch_003.pt"
```

Nếu resume từ `epoch_003.pt`, chương trình sẽ chạy tiếp:

```text
epoch 4
epoch 5
epoch 6
```

Checkpoint lưu cả trạng thái của **Adam – bộ tối ưu**, vì vậy có thể tiếp tục huấn luyện từ trạng thái trước đó.

Nếu muốn train hoàn toàn lại từ đầu:

```python
MODE = "new"
```

và chạy notebook từ đầu.
